# Notebook 15 — E27: quatro famílias sobre relabel-15 + sub11

**E26 → E27.** Testa, no protocolo fixo (`StratifiedKFold(5, shuffle, seed=42)` em
`train_nb02.csv`, Macro-F1, validação sempre com rótulos originais):
1. **mc5** — XGBoost multiclasse (receita nb11) com rótulos *relabel-15*, 5 sementes;
2. **famílias do colega** (NB03 `MODELOS_ORDINAIS`): LGBM×4 e HGB×2 como F&H-15 bins;
3. combinações **iguais-peso** (sem tuning): A {FH,mc5}, **B1 {FH,LGB,HGB,mc5}**, B3;
4. regra de adoção: **5/5 folds vs E26 (0.5364)** → **B1 vence: 0.5460 → sub11**.

Caches `data/nb15_*` — reruns retomam. Resultados: `data/nb15_resultados.csv`.

In [1]:
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE, K, TARGET = 42, 16, 'AGE_RANGE'

def _data_dir():
    for cand in [os.path.join('..', '..', 'data'), os.path.join('..', 'data'), 'data']:
        if os.path.isdir(cand): return cand
    raise FileNotFoundError('pasta data/ nao encontrada')
DATA_DIR = _data_dir()
SUB_DIR = os.path.join(os.path.dirname(DATA_DIR), 'submissions')

tr = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
y = tr[TARGET].to_numpy().astype(int)
Xv = tr.drop(columns=['PID', TARGET]).to_numpy(dtype=np.float64)
fmin = tr['FAIXA_MINIMA'].to_numpy().astype(int)
n = len(y)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(Xv, y))

# rótulos relabel-15 (cross-fit nb11, sem seleção por rótulo)
d11 = np.load(os.path.join(DATA_DIR, 'nb11_oof311.npz'))
oof_x, proba_x = d11['oof'].astype(int), d11['proba']
rank = np.argsort(proba_x[np.arange(n), y])
y_lab = y.copy(); rr = rank[:int(n * 0.15)]; y_lab[rr] = oof_x[rr]
E26 = np.array([0.5464, 0.5460, 0.5324, 0.5284, 0.5288])   # NB13 argmax+corr
RESULTADOS = []

def show(exp, nome, f, agg=None):
    d = f - E26
    r = {'exp': exp, 'variante': nome, 'f1': round(f.mean(), 4), 'std': round(f.std(), 4),
         'delta_vs_E26': round(d.mean(), 4), 'folds_ganhos': int((d > 0).sum())}
    RESULTADOS.append(r)
    print(f'{nome:<34} {f.mean():.4f} +/- {f.std():.4f} folds={np.round(f,4).tolist()} '
          f'delta {d.mean():+.4f} wins {int((d>0).sum())}/5' + (f' agg {agg:.4f}' if agg else ''))
    return r

def folds_f1(P, corr=True, med=False):
    arg = P.argmax(axis=1)
    if med:
        m = np.array([int(np.median(np.repeat(np.arange(K), (P[i]*1000).astype(int)))) for i in range(n)])
        arg = np.where(P.max(axis=1) < 0.40, m, arg)
    if corr: arg = np.maximum(arg, fmin)
    return np.array([f1_score(y[va], arg[va], average='macro') for _, va in FOLDS]), arg

## 1. mc5 — multiclasse + relabel-15, 5 sementes

In [2]:
def xgb_recipe(seed):
    return XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist', random_state=seed,
        n_jobs=-1, eval_metric='mlogloss')

p = os.path.join(DATA_DIR, 'nb15_mc5.npz')
if os.path.exists(p):
    Pall_m = np.load(p)['Pall']; print('mc5: cache')
else:
    Pall_m = np.zeros((5, n, K)); t0 = time.time()
    for fi, (a, b) in enumerate(FOLDS):
        sw = compute_sample_weight('balanced', y_lab[a])
        for si, sd in enumerate([42, 1, 2, 3, 4]):
            m = xgb_recipe(sd); m.fit(Xv[a], y_lab[a], sample_weight=sw)
            Pall_m[si, b] = m.predict_proba(Xv[b])
        print(f'  fold {fi} ({time.time()-t0:.0f}s)', flush=True)
    np.savez_compressed(p, Pall=Pall_m)
MC5 = Pall_m.mean(axis=0)
f_mc, _ = folds_f1(MC5)
show('E26', 'mc5 multiclasse+relabel 5seed', f_mc, f1_score(y, np.maximum(MC5.argmax(1), fmin), average='macro'))

mc5: cache
mc5 multiclasse+relabel 5seed      0.5376 +/- 0.0041 folds=[0.5395, 0.5433, 0.5389, 0.5348, 0.5315] delta +0.0012 wins 3/5 agg 0.5377


{'exp': 'E26',
 'variante': 'mc5 multiclasse+relabel 5seed',
 'f1': np.float64(0.5376),
 'std': np.float64(0.0041),
 'delta_vs_E26': np.float64(0.0012),
 'folds_ganhos': 3}

## 2. Famílias ordinais do colega (NB03) sobre relabel-15

In [3]:
def pfull_from_acima(acima):
    acima = np.minimum.accumulate(acima, axis=1)
    proba = np.column_stack([1 - acima[:, 0], acima[:, :-1] - acima[:, 1:], acima[:, -1]])
    proba = np.clip(proba, 1e-6, None)
    return proba / proba.sum(axis=1, keepdims=True)

MAKERS = {
 'lgbm': (lambda s: LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
            min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
            reg_lambda=1, verbose=-1, n_jobs=-1, random_state=s), [42, 1, 2, 3]),
 'hgb':  (lambda s: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
            max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1,
            early_stopping=False, random_state=s), [42, 1]),
}

def run_family(tag, maker, seeds):
    p = os.path.join(DATA_DIR, f'nb15_fam_{tag}.npz')
    if os.path.exists(p):
        print(f'{tag}: cache'); return np.load(p)['Pall']
    Pall = np.zeros((len(seeds), n, K)); t0 = time.time()
    for fi, (a, b) in enumerate(FOLDS):
        sw = compute_sample_weight('balanced', y_lab[a])
        for si, sd in enumerate(seeds):
            acima = np.zeros((len(b), 15))
            for k in range(15):
                m = maker(sd)
                m.fit(Xv[a], (y_lab[a] > k).astype(int), sample_weight=sw)
                acima[:, k] = m.predict_proba(Xv[b])[:, 1]
            Pall[si, b] = pfull_from_acima(acima)
        print(f'  {tag} fold {fi} ({time.time()-t0:.0f}s)', flush=True)
    np.savez_compressed(p, Pall=Pall)
    return Pall

Px = np.load(os.path.join(DATA_DIR, 'nb13_rel15_5seed.npz'))['Pall']   # F&H×5 (E26)
Pl = run_family('lgbm', *MAKERS['lgbm'])
Ph = run_family('hgb',  *MAKERS['hgb'])
fx, _ = folds_f1(Px.mean(axis=0)); show('E26', 'XGBx5 F&H (ref E26)', fx)
fl, _ = folds_f1(Pl.mean(axis=0)); show('fam', 'LGBMx4 sozinho', fl, f1_score(y, np.maximum(Pl.mean(0).argmax(1), fmin), average='macro'))
fh, _ = folds_f1(Ph.mean(axis=0)); show('fam', 'HGBx2 sozinho', fh)
ff, pred11 = folds_f1(np.concatenate([Px, Pl, Ph], axis=0).mean(axis=0))
show('fam', 'media plana11 (estilo sub7)', ff, f1_score(y, pred11, average='macro'))
fb, _ = folds_f1(np.mean([Px.mean(0), Pl.mean(0), Ph.mean(0)], axis=0))
show('fam', 'media balanceada por familia', fb)
fm, _ = folds_f1(np.concatenate([Px, Pl, Ph], axis=0).mean(axis=0), med=True)
show('fam', 'plana11 + mediana@0.40', fm)

lgbm: cache
hgb: cache
XGBx5 F&H (ref E26)                0.5364 +/- 0.0081 folds=[0.5464, 0.546, 0.5324, 0.5284, 0.5288] delta -0.0000 wins 2/5
LGBMx4 sozinho                     0.5417 +/- 0.0099 folds=[0.5495, 0.5546, 0.5426, 0.5354, 0.5266] delta +0.0053 wins 4/5 agg 0.5419
HGBx2 sozinho                      0.5355 +/- 0.0059 folds=[0.5445, 0.5367, 0.532, 0.5373, 0.5268] delta -0.0009 wins 1/5


media plana11 (estilo sub7)        0.5409 +/- 0.0102 folds=[0.5459, 0.5554, 0.5437, 0.5335, 0.526] delta +0.0045 wins 3/5 agg 0.5411
media balanceada por familia       0.5401 +/- 0.0088 folds=[0.5448, 0.5526, 0.5413, 0.5351, 0.5266] delta +0.0037 wins 3/5


plana11 + mediana@0.40             0.5395 +/- 0.0104 folds=[0.5458, 0.5555, 0.5387, 0.5308, 0.5265] delta +0.0031 wins 3/5


{'exp': 'fam',
 'variante': 'plana11 + mediana@0.40',
 'f1': np.float64(0.5395),
 'std': np.float64(0.0104),
 'delta_vs_E26': np.float64(0.0031),
 'folds_ganhos': 3}

## 3. Combinações iguais-peso (sem tuning) → adoção 5/5

In [4]:
combos = {'A {FH,mc5}': (Px.mean(0) + MC5) / 2,
          'B1 {FH,LGB,HGB,mc5} igual-peso': (Px.mean(0) + Pl.mean(0) + Ph.mean(0) + MC5) / 4,
          'B3 {FH,LGB,mc5}': (Px.mean(0) + Pl.mean(0) + MC5) / 3}
vencedor, melhor = None, -1
for nome, P in combos.items():
    f, pred = folds_f1(P)
    r = show('E27', nome, f, f1_score(y, pred, average='macro'))
    if r['folds_ganhos'] == 5 and r['f1'] > melhor:
        vencedor, melhor, Pv = nome, r['f1'], P
print('ADOPT-5/5:', vencedor)
assert vencedor == 'B1 {FH,LGB,HGB,mc5} igual-peso', 'a receita vencedora mudou — rever sub11'
pd.DataFrame(RESULTADOS).to_csv(os.path.join(DATA_DIR, 'nb15_resultados.csv'), index=False)
print(pd.DataFrame(RESULTADOS).to_string(index=False))

A {FH,mc5}                         0.5448 +/- 0.0065 folds=[0.5487, 0.5537, 0.5468, 0.5371, 0.5376] delta +0.0084 wins 5/5 agg 0.5447
B1 {FH,LGB,HGB,mc5} igual-peso     0.5460 +/- 0.0079 folds=[0.5547, 0.5544, 0.5467, 0.5347, 0.5395] delta +0.0096 wins 5/5 agg 0.5463
B3 {FH,LGB,mc5}                    0.5446 +/- 0.0063 folds=[0.5478, 0.5531, 0.5473, 0.5357, 0.539] delta +0.0082 wins 5/5 agg 0.5448
ADOPT-5/5: B1 {FH,LGB,HGB,mc5} igual-peso
exp                       variante     f1    std  delta_vs_E26  folds_ganhos
E26  mc5 multiclasse+relabel 5seed 0.5376 0.0041        0.0012             3
E26            XGBx5 F&H (ref E26) 0.5364 0.0081       -0.0000             2
fam                 LGBMx4 sozinho 0.5417 0.0099        0.0053             4
fam                  HGBx2 sozinho 0.5355 0.0059       -0.0009             1
fam    media plana11 (estilo sub7) 0.5409 0.0102        0.0045             3
fam   media balanceada por familia 0.5401 0.0088        0.0037             3
fam         plana1

## 4. sub11 — teste com a receita E27 (caches por família)

In [5]:
te = pd.read_csv(os.path.join(DATA_DIR, 'test_nb02.csv'))
Xte = te.drop(columns=['PID']).to_numpy(dtype=np.float64)
fmin_te = te['FAIXA_MINIMA'].to_numpy().astype(int)
assert len(Xte) == 3655
sw = compute_sample_weight('balanced', y_lab)

def mean_over(cache_fmt, maker, seeds, multi=False):
    outs = []
    for sd in seeds:
        p = os.path.join(DATA_DIR, cache_fmt.format(sd=sd))
        if os.path.exists(p):
            outs.append(np.load(p)['pf']); print(f'  seed {sd}: cache'); continue
        t0 = time.time()
        if multi:
            m = xgb_recipe(sd); m.fit(Xv, y_lab, sample_weight=sw); pf = m.predict_proba(Xte)
        else:
            acima = np.zeros((len(Xte), 15))
            for k in range(15):
                m = maker(sd); m.fit(Xv, (y_lab > k).astype(int), sample_weight=sw)
                acima[:, k] = m.predict_proba(Xte)[:, 1]
            pf = pfull_from_acima(acima)
        np.savez_compressed(p, pf=pf); outs.append(pf)
        print(f'  seed {sd} done ({time.time()-t0:.0f}s)', flush=True)
    return np.mean(outs, axis=0)

fh_te = np.mean([np.load(os.path.join(DATA_DIR, f'nb13_rel15_te_s{sd}.npz'))['pf']
                 for sd in [42, 1, 2, 3, 4]], axis=0)
lgb_te = mean_over('nb15_lgb_te_s{sd}.npz', MAKERS['lgbm'][0], [42, 1, 2, 3])
hgb_te = mean_over('nb15_hgb_te_s{sd}.npz', MAKERS['hgb'][0], [42, 1])
mc5_te = mean_over('nb15_mc5_te_s{sd}.npz', None, [42, 1, 2, 3, 4], multi=True)
Pte = (fh_te + lgb_te + hgb_te + mc5_te) / 4
pred = np.maximum(Pte.argmax(axis=1), fmin_te).astype(int)
assert len(pred) == 3655 and pred.min() >= 0 and pred.max() <= 15
out = os.path.join(SUB_DIR, 'sub11_4familias_rel15.csv')
pd.DataFrame({'RowId': np.arange(len(pred)), TARGET: pred}).to_csv(out, index=False)
print('gravado:', out, pd.Series(pred).value_counts().sort_index().to_dict())

  seed 42: cache
  seed 1: cache
  seed 2: cache
  seed 3: cache
  seed 42: cache
  seed 1: cache


  seed 42: cache
  seed 1: cache
  seed 2: cache
  seed 3: cache
  seed 4: cache
gravado: ../submissions/sub11_4familias_rel15.csv {0: 70, 1: 41, 2: 142, 3: 145, 4: 168, 5: 196, 6: 286, 7: 308, 8: 271, 9: 360, 10: 367, 11: 330, 12: 371, 13: 263, 14: 217, 15: 120}
